# Конспект: Логистическая регрессия и метрики классификации

## 01. Логистическая регрессия

### Что такое логистическая регрессия?
**Логистическая регрессия (Logistic Regression)** — это алгоритм машинного обучения с учителем, который, несмотря на слово «регрессия» в названии, используется для решения задач **классификации**. Суть алгоритма заключается в том, что он оценивает вероятность принадлежности объекта к определенному классу (от $0$ до $1$), а затем на основе этой вероятности относит объект к конкретной категории.

### Для каких задач используется?
Алгоритм применяется там, где нужно отнести объект к одной из заранее известных категорий:
* **Медицина:** определение, болен ли пациент (да / нет) по результатам анализов.
* **Банковское дело (кредитный скоринг):** одобрить или отказать в кредите (вернет / не вернет).
* **Маркетинг и IT:** уйдет ли клиент к конкурентам (отток), является ли письмо спамом, кликнет ли пользователь по рекламному баннеру.

### Чем отличается от линейной регрессии?
| Критерий | Линейная регрессия (Linear Regression) | Логистическая регрессия (Logistic Regression) |
| :--- | :--- | :--- |
| **Тип задачи** | Регрессия (предсказание непрерывного числа) | Классификация (предсказание категории/класса) |
| **Выходное значение** | Любое число от $-\infty$ до $+\infty$ (например, цена квартиры, температура) | Вероятность в диапазоне строго от $0$ до $1$ |
| **Геометрический смысл** | Прямая линия (гиперплоскость), которая пытается пройти максимально близко ко всем точкам | S-образная кривая (сигмоида), которая разделяет пространство на области классов |
| **Функция потерь** | MSE (среднеквадратичная ошибка) | Log-Loss (бинарная кросс-энтропия) |

### Что такое бинарная классификация?
**Бинарная классификация** — это задача разделения объектов строго на **два взаимоисключающих класса**, которые обычно кодируются как `0` (отрицательный класс) и `1` (положительный класс).
* *Примеры:* «Спам (1) / Не спам (0)», «Мошенническая транзакция (1) / Обычная транзакция (0)».

### Что такое Sigmoid Function (Сигмоида)?
**Сигмоида** — это математическая S-образная функция, которая берет на вход любое вещественное число (от $-\infty$ до $+\infty$), полученное от линейной комбинации признаков $z = w_0 + w_1x_1 + \dots + w_nx_n$, и «сжимает» его в диапазон от $0$ до $1$. Благодаря этому выход функции можно интерпретировать как **вероятность**.

**Формула сигмоиды:**
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
где:
* $z$ — результат линейного уравнения ($w^T x + b$),
* $e$ — число Эйлера ($\approx 2.718$).

### Как модель определяет принадлежность к классу?
1. Модель вычисляет взвешенную сумму признаков объекта ($z$).
2. Пропускает $z$ через сигмоиду и получает вероятность $P(y=1)$ того, что объект принадлежит к классу `1`.
3. Сравнивает полученную вероятность с **порогом классификации (Threshold)** — по умолчанию он равен **$0.5$**:
   * Если $P(y=1) \ge 0.5 \rightarrow$ модель предсказывает класс **`1`**.
   * Если $P(y=1) < 0.5 \rightarrow$ модель предсказывает класс **`0`**.

In [ ]:
import numpy as np


def sigmoid(z):
  return 1 / (1 + np.exp(-z))


# Проверим работу на нескольких значениях z
for z_val in [-3.0, 0.0, 1.5]:
  prob = sigmoid(z_val)
  pred_class = int(prob >= 0.5)
  print(f"z = {z_val:4.1f} | Вероятность = {prob:.3f} | Класс = {pred_class}")

## 02. Метрики классификации

### Confusion Matrix (Матрица ошибок)
Это таблица размером $2 \times 2$ (для бинарной классификации), которая показывает, сколько раз модель угадала, а сколько раз ошиблась, в разбивке по классам.

| | Предсказано: **1 (Positive)** | Предсказано: **0 (Negative)** |
| :--- | :---: | :---: |
| **На самом деле: 1 (Positive)** | **TP** (True Positive) | **FN** (False Negative) |
| **На самом деле: 0 (Negative)** | **FP** (False Positive) | **TN** (True Negative) |

* **True Positive (TP — Истинно положительный):** объект действительно класса `1`, и модель правильно предсказала `1`. *(Пример: пациент болен, и тест показал болезнь).*
* **True Negative (TN — Истинно отрицательный):** объект класса `0`, и модель правильно предсказала `0`. *(Пример: пациент здоров, и тест показал, что он здоров).*
* **False Positive (FP — Ложноположительный / Ошибка I рода):** объект на самом деле класса `0`, но модель ошибочно предсказала `1` (ложная тревога). *(Пример: пациент здоров, но тест ошибочно показал болезнь).*
* **False Negative (FN — Ложноотрицательный / Ошибка II рода):** объект на самом деле класса `1`, но модель ошибочно предсказала `0` (пропуск цели). *(Пример: пациент болен, но тест этого не заметил).*

---

### Основные метрики

#### 1. Accuracy (Доля правильных ответов)
* **Объяснение:** Показывает, какую долю от всех объектов модель классифицировала верно.
* **Формула:**
  $$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
* **Пример использования:** Оценка качества модели распознавания рукописных цифр или классификации фотографий кошек и собак, где количество объектов каждого класса примерно одинаково.

#### 2. Precision (Точность)
* **Объяснение:** Показывает, какая доля объектов, которые модель назвала положительными (`1`), действительно являются положительными. Метрика отвечает на вопрос: *«Насколько можно доверять модели, когда она говорит "Да"?»* (штрафует за ложные срабатывания — **FP**).
* **Формула:**
  $$\text{Precision} = \frac{TP}{TP + FP}$$
* **Пример использования:** Спам-фильтр электронной почты. Важно не отправить важное рабочее письмо (класс `0`) в папку «Спам» (класс `1`), то есть нужно минимизировать **FP**.

#### 3. Recall (Полнота)
* **Объяснение:** Показывает, какую долю реальных положительных объектов модель смогла обнаружить из всех существующих. Метрика отвечает на вопрос: *«Какой процент всех целей модель сумела поймать?»* (штрафует за пропуски — **FN**).
* **Формула:**
  $$\text{Recall} = \frac{TP}{TP + FN}$$
* **Пример использования:** Диагностика онкологических заболеваний. Нужно выявить как можно больше реально больных пациентов (минимизировать **FN**), даже если из-за этого придется отправить на дополнительное обследование пару здоровых людей.

#### 4. F1-Score (F1-мера)
* **Объяснение:** Это среднее гармоническое между Precision и Recall. Она объединяет обе метрики в одно число и помогает найти баланс между ними. F1-Score будет высоким только тогда, когда и Precision, и Recall высоки одновременно (если одна из метрик близка к нулю, F1 тоже упадет к нулю).
* **Формула:**
  $$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$
* **Пример использования:** Поисковые системы или банковский антифрод (поиск мошеннических транзакций), где мошенников мало (дисбаланс классов), но важно и не пропускать воров (Recall), и не блокировать карты честным клиентам (Precision).

In [1]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# Тестовые данные (например: 1 - спам, 0 - не спам)
y_true = [1, 0, 1, 1, 0, 0, 1, 0, 0, 0]  # Реальные метки
y_pred = [1, 0, 1, 0, 1, 0, 1, 0, 0, 0]  # Предсказания модели

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print(f"\nTP = {tp}, TN = {tn}, FP = {fp}, FN = {fn}")
print("-" * 30)
print(f"Accuracy:  {accuracy_score(y_true, y_pred):.2f}")
print(f"Precision: {precision_score(y_true, y_pred):.2f}")
print(f"Recall:    {recall_score(y_true, y_pred):.2f}")
print(f"F1-Score:  {f1_score(y_true, y_pred):.2f}")

Confusion Matrix:
[[5 1]
 [1 3]]

TP = 3, TN = 5, FP = 1, FN = 1
------------------------------
Accuracy:  0.80
Precision: 0.75
Recall:    0.75
F1-Score:  0.75


## 03. Вопросы для размышления

### 1. Почему Accuracy не всегда является хорошей метрикой?
Accuracy перестает работать при **дисбалансе классов** — когда одного класса в данных намного больше, чем другого.
* *Пример:* Представим данные по банковским транзакциям, где $99.9\%$ операций честные (`0`), и только $0.1\%$ — мошеннические (`1`). Если написать «глупую» модель, которая всегда отвечает `0` (все честные), её **Accuracy составит $99.9\%$**. На бумаге цифра выглядит идеально, но на практике модель абсолютно бесполезна, так как она не поймала ни одного мошенника ($\text{Recall} = 0$).

### 2. В чём разница между Precision и Recall?
Разница заключается в том, **какую ошибку мы считаем более дорогой**:
* **Precision** фокусируется на качестве положительных предсказаний и наказывает за **False Positive** (ложные тревоги).
* **Recall** фокусируется на охвате реальных положительных случаев и наказывает за **False Negative** (пропуски).
* Между ними существует компромисс (Precision-Recall Trade-off): если снизить порог классификации (например, с $0.5$ до $0.2$), модель станет чаще говорить «Да» — **Recall вырастет**, но из-за увеличения числа ложных срабатываний **Precision упадет**.

### 3. В каких ситуациях Precision важнее Recall и наоборот?
* **Precision важнее Recall (цена ошибки FP выше, чем FN):**
  1. **Судебная система / Система автоштрафов:** Лучше не выписать штраф нарушителю (FN), чем несправедливо наказать невиновного человека (FP).
  2. **Рекомендации контента для детей:** Важно, чтобы в подборку с пометкой «безопасно» попадали только проверенные видео (максимальный Precision), даже если часть хороших роликов не пройдет фильтр.
* **Recall важнее Precision (цена ошибки FN выше, чем FP):**
  1. **Системы безопасности в аэропорту (поиск взрывчатки/оружия):** Лучше лишний раз досмотреть чемодан обычного туриста (FP), чем пропустить на борт самолета реальную угрозу (FN).
  2. **Обнаружение поломок на производстве или АЭС:** Дешевле провести внеплановую проверку исправного датчика (FP), чем пропустить реальную аварию (FN).